## load the Deep Fashion dataset

In [ ]:
# the relative imports work if the module is a part of a package, and the script is being executed 
# as a module and this doesn't apply on jupyter notebook

# so we need to add the parent folder to sys.path
import sys 
import os 
import mediapipe as mp
import cv2
import json
import pandas as pd 
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
path = os.path.abspath(Path(os.getcwd()).parent)
print(path)

d:\03. Projects\technocolabs_projects\1. Deep-Learning-Driven-Virtual-Fashion-Fitting


In [59]:
# import this path to sys.path
sys.path.append(path)

from preprocessing import config as C


### count the number of images 

In [60]:
def count_images(dir_path: Path):
    exts = ("*.jpg", "*.jpeg", "*.png")
    return sum(len(list(dir_path.rglob(ext))) for ext in exts)

deep_fashion_images = C.DEEP_FASHION_DIR
deep_fashion_annotations = C.ANNOTATIONS_DIR

fashion_gen_images = C.FASHION_GEN_DIR
fashion_gen_annotation = C.STYLES_GEN_DIR

pose_simples = C.POSE_DIR / "Simples"


print("Checking number of images in each Directory\n" + "-" * 40)
print("\nCounting images:\n" + "-"*40)
print(f"DeepFashion images    : {count_images(deep_fashion_images)}")
print(f"Fashion-Gen images     : {count_images(fashion_gen_images)}")
print(f"Pose sample images     : {count_images(pose_simples)}")

Checking number of images in each Directory
----------------------------------------

Counting images:
----------------------------------------
DeepFashion images    : 289219
Fashion-Gen images     : 44441
Pose sample images     : 2


### mediapipe pose test 

In [61]:
import cv2
import mediapipe as mp
import json
print("Running a quick mediapipe test ...")
img_path = next(pose_simples.glob("*.jpg"), None)
if img_path is None:
    print('There is no images ===> skipping pose test')
    
image = cv2.imread(str(img_path))
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

with mp_pose.Pose(static_image_mode=True) as pose:
    results = pose.process(image_rgb)
    
    if results.pose_landmarks:
        print(f"detected {len(results.pose_landmarks.landmark)} keypoints")
        
        # drawing landmarks on a copy of image
        annotated_image = image.copy()
        mp_drawing.draw_landmarks(
            annotated_image,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )
        image_out_path = img_path.with_name(f"{img_path.stem}_pose.jpg")
        cv2.imwrite(str(image_out_path), annotated_image)
        
        # save the json data
        keypoints = [
            {
                "x": round(lmk.x, 5),
                "y": round(lmk.y, 5),
                "z": round(lmk.z, 5),
                "visibility": round(lmk.visibility, 5) 
            }
            for lmk in results.pose_landmarks.landmark
        ]
        
        json_path_out = img_path.with_name(f"{img_path.stem}_pose.json")
        with open(json_path_out, "w") as f:
            json.dump({"file": img_path.name, "landmarks": keypoints}, f, indent=2)

Running a quick mediapipe test ...
detected 33 keypoints


### load category images

In [62]:
category_images = pd.read_csv(
    C.ANNOTATIONS_DIR / "category" /"list_category_img.txt",
    sep = r"\s+",
    header= None,
    skiprows= 2,
    names= ['file', 'category_label']
)

In [63]:
category_images.head()

,file,category_label
0,img/Sheer_Pleated-Front_Blouse/img_00000001.jpg,3
1,img/Sheer_Pleated-Front_Blouse/img_00000002.jpg,3
2,img/Sheer_Pleated-Front_Blouse/img_00000003.jpg,3
3,img/Sheer_Pleated-Front_Blouse/img_00000004.jpg,3
4,img/Sheer_Pleated-Front_Blouse/img_00000005.jpg,3


In [64]:
print(category_images["category_label"].unique())

[ 3  2  5  4  1 18 17 19 16  7 10  6 11  9 15 12 20 13 14  8 33 32 26 29
 34 27 24 35 30 23 22 36 31 25 28 21 41 48 39 44 42 47 37 43 40 46]


### load categorical cloth

In [65]:
category_cloth = pd.read_csv(
    C.ANNOTATIONS_DIR / "category" / "list_category_cloth.txt",
    sep = r"\s+",
    skiprows= 2,
    header= None,
    names= ['category_name', 'category_type']
).reset_index(names="category_label") #the category_label is the index of the category_cloth now

# add the region column
category_cloth["region"] = category_cloth["category_type"].map({1: "upper", 2: "lower", 3: "full"})

category_cloth.head()

,category_label,category_name,category_type,region
0,0,Anorak,1,upper
1,1,Blazer,1,upper
2,2,Blouse,1,upper
3,3,Bomber,1,upper
4,4,Button-Down,1,upper


In [66]:
category_cloth.shape

(50, 4)

### merge the two datasets 


In [67]:
df = category_images.merge(category_cloth, on="category_label", how="left")

df.head()

,file,category_label,category_name,category_type,region
0,img/Sheer_Pleated-Front_Blouse/img_00000001.jpg,3,Bomber,1,upper
1,img/Sheer_Pleated-Front_Blouse/img_00000002.jpg,3,Bomber,1,upper
2,img/Sheer_Pleated-Front_Blouse/img_00000003.jpg,3,Bomber,1,upper
3,img/Sheer_Pleated-Front_Blouse/img_00000004.jpg,3,Bomber,1,upper
4,img/Sheer_Pleated-Front_Blouse/img_00000005.jpg,3,Bomber,1,upper


### add the bbox coordinate and the split column

In [68]:

# 4. add the bbox coordinate
bbox_coordinate = pd.read_csv(
    C.ANNOTATIONS_DIR / "bbox" / "list_bbox.txt",
    sep=r"\s+",
    header=None,
    skiprows=2,
    names = ["file", "x_1", "y_1", "x_2", "y_2"]
)

df = df.merge(bbox_coordinate,on="file" ,how="left")

# 5. add the split info
split = pd.read_csv(
    C.ANNOTATIONS_DIR  / "splits" / "list_eval_partition.txt",
    sep=r"\s+", header=None, skiprows=2,
    names=["file", "split"]
)
df = df.merge(split, on="file", how="left")

df = df.rename(columns={"category_name": "category"})
df.head()

,file,category_label,category,category_type,region,x_1,y_1,x_2,y_2,split
0,img/Sheer_Pleated-Front_Blouse/img_00000001.jpg,3,Bomber,1,upper,72,79,232,273,train
1,img/Sheer_Pleated-Front_Blouse/img_00000002.jpg,3,Bomber,1,upper,67,59,155,161,train
2,img/Sheer_Pleated-Front_Blouse/img_00000003.jpg,3,Bomber,1,upper,65,65,156,200,val
3,img/Sheer_Pleated-Front_Blouse/img_00000004.jpg,3,Bomber,1,upper,51,62,167,182,train
4,img/Sheer_Pleated-Front_Blouse/img_00000005.jpg,3,Bomber,1,upper,46,88,166,262,test


In [69]:
df["split"].value_counts() / len(df) * 100 

split
train    72.339587
val      13.830207
test     13.830207
Name: count, dtype: float64

In [70]:
df.describe()

,category_label,category_type,x_1,y_1,x_2,y_2
count,289222.000000,289222.000000,289222.000000,289222.000000,289222.000000,289222.000000
mean,25.210990,1.831870,47.561603,49.928470,189.198567,245.724122
std,14.025825,0.876754,36.751532,37.568463,48.065077,52.651559
min,1.000000,1.000000,1.000000,1.000000,26.000000,15.000000
25%,16.000000,1.000000,21.000000,23.000000,156.000000,209.000000
50%,26.000000,2.000000,44.000000,48.000000,182.000000,261.000000
75%,41.000000,3.000000,67.000000,70.000000,215.000000,293.000000
max,48.000000,3.000000,264.000000,268.000000,300.000000,300.000000


## load the Fashion Gen dataset 

In [71]:
# load the styles.csv file 

styles = pd.read_csv(C.STYLES_GEN_DIR, on_bad_lines='warn', engine="python")

# rename the id to file and subCategory to category
styles = styles.rename(columns={"id":"file", "subCategory" : "category"})

In [72]:
# missing category rows 
styles["category"].isnull().sum()


0

### add the .jpg extension to file name 

In [73]:
styles["file"] = styles["file"].astype(str) + ".jpg"

In [74]:
styles.head()

,file,gender,masterCategory,category,articleType,baseColour,season,year,usage,productDisplayName
0,15970.jpg,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011,Casual,Turtle Check Men Navy Blue Shirt
1,39386.jpg,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012,Casual,Peter England Men Party Blue Jeans
2,59263.jpg,Women,Accessories,Watches,Watches,Silver,Winter,2016,Casual,Titan Women Silver Watch
3,21379.jpg,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011,Casual,Manchester United Men Solid Black Track Pants
4,53759.jpg,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012,Casual,Puma Men Grey T-shirt


### add a split column

we will use 80-10-10 logic:

    - 0==>7 : train
    - 8: val
    - 9: test

In [75]:
styles["split"] = (
    styles.groupby("category").cumcount()
    .apply(lambda i : "train" if i%10 <8 else ("val" if i%10 == 8 else "test"))
)

styles.head()

,file,gender,masterCategory,category,articleType,baseColour,season,year,usage,productDisplayName,split
0,15970.jpg,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011,Casual,Turtle Check Men Navy Blue Shirt,train
1,39386.jpg,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012,Casual,Peter England Men Party Blue Jeans,train
2,59263.jpg,Women,Accessories,Watches,Watches,Silver,Winter,2016,Casual,Titan Women Silver Watch,train
3,21379.jpg,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011,Casual,Manchester United Men Solid Black Track Pants,train
4,53759.jpg,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012,Casual,Puma Men Grey T-shirt,train


In [76]:
styles["split"].value_counts() * 100 / len(styles)

split
train    80.085089
val       9.963083
test      9.951828
Name: count, dtype: float64

### clean the pose data

In [77]:
from preprocessing.clean_pose import clean_pose_data

In [78]:
path = r"D:\03. Projects\technocolabs_projects\1. Deep-Learning-Driven-Virtual-Fashion-Fitting\Pose\Simples"
clean_pose_data(path)